In [1]:
# !pip install git+https://github.com/monash-emu/summer3wip.git

In [2]:
import numpy as np
from plotly import graph_objects as go

import pandas as pd
pd.options.plotting.backend = "plotly"

from summer3.graph import defer, Parameter, CompartmentValues
from summer3.epi import Stratification, CompartmentMap, \
    ManagedArray, CategoryData, TransitionFlow, CompartmentalEpiModel, \
    build_istate, dti_to_epoch

In [3]:
population = 1e6  
seed = 1.0
start_time = 0
end_time = 50
model_comps = ["susceptible", "infectious", "recovered"]
infect_comps = ["infectious"]
disease_state = Stratification("disease_state", model_comps)
humans = CompartmentMap.new(disease_state)
epi_model = CompartmentalEpiModel(humans, range(start_time, end_time))
start_pop = [population - seed, seed, 0.0]
epi_model.set_initial_population(CategoryData(disease_state.categories(), np.array(start_pop)))

def iprocess(compartment_values: ManagedArray, contact_rate: float, infectious_compartments: tuple):
    n_infectious = compartment_values.query(infectious_compartments).sum()
    n_population = compartment_values.sum()
    infectious_prevalence = n_infectious / n_population
    return contact_rate * infectious_prevalence

foi = defer(iprocess)(CompartmentValues, Parameter("effective_contact_rate", 0.0), disease_state["infectious"])
infection = TransitionFlow("infection", disease_state["susceptible"], disease_state["infectious"], foi)
epi_model.add_flow(infection)

recovery = TransitionFlow("recovery", disease_state["infectious"], disease_state["recovered"], Parameter("recovery_rate", 0.0))
epi_model.add_flow(recovery)
base_parameters = {"effective_contact_rate": 1.2, "recovery_rate": 0.2}